# panjika

> the register of deeds

Which agent session changed this file? Did the change stay?

Each repository has one ledger. It is a JSONL file, and panjika only adds to it. Each harness writes to the same ledger.

In [ ]:
#| hide
from panjika.core import *
from panjika.git import *
from panjika.harness import *

In [ ]:
#| hide
import subprocess, tempfile, time
from pathlib import Path

def git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'billing'; d.mkdir(parents=True)
git(d, 'init', '-q', '-b', 'main')
git(d, 'config', 'user.email', 'sam@example.com'); git(d, 'config', 'user.name', 'Sam')
Home(d/'.panjika').init()

A billing module in git.

In [ ]:
charges = d/'charges.py'
charges.write_text('def total(rows):\n    return sum(r["amount"] for r in rows)\n')
git(d, 'add', '-A'); git(d, 'commit', '-qm', 'the billing module')
print(charges.read_text())

Monday. Claude Code sends one hook payload for each event.

In [ ]:
def cc(**kw): ingest({'session_id': 'cc-4f21', 'cwd': str(d), **kw}, 'claude-code', d/'.panjika')

cc(hook_event_name='SessionStart', model='opus-5')
cc(hook_event_name='UserPromptSubmit',
   user_input='total() blows up on refunds. Make it skip negative amounts.')

charges.write_text('def total(rows):\n    return sum(r["amount"] for r in rows if r["amount"] > 0)\n')

cc(hook_event_name='PostToolUse', tool_name='Edit',
   tool_input={'file_path': str(charges)}, tool_response='ok')
cc(hook_event_name='PostToolUseFailure', tool_name='Bash',
   tool_input={'command': 'pytest -q'}, tool_response='1 failed')
cc(hook_event_name='SessionEnd', reason='clear')

git(d, 'commit', '-aqm', 'skip negative amounts in total()')
link_commit('HEAD', home=d/'.panjika', start=d)
print(charges.read_text())

The `post-commit` hook runs `link_commit`. The commit finds its session, even if a person ran git.

Wednesday. Ramabana gives the full turn record that `Agent._remember` makes.

In [ ]:
time.sleep(1.1)
charges.write_text(
    'def total(rows):\n    return round(sum(r["amount"] for r in rows if r["amount"] > 0), 2)\n')

ingest({'session': 'rb-0c7e', 'cwd': str(d), 'at': time.time(), 'model': 'sonnet',
        'prompt': 'total() should round to 2dp', 'reply': 'rounded it',
        'usage': {'input': 1840, 'output': 96},
        'activity': [{'tool': 'edit_file', 'args': {'path': str(charges)},
                      'ok': True, 'secs': 0.5}]},
       'ramabana', d/'.panjika')

git(d, 'commit', '-aqm', 'round the total to 2dp')
print(charges.read_text())

One hour later Sam does not agree. He gives the reason in the commit message.

In [ ]:
time.sleep(1.1)
charges.write_text('def total(rows):\n    return sum(r["amount"] for r in rows if r["amount"] > 0)\n')
git(d, 'commit', '-aqm', 'no rounding: amounts are already in cents')
print(charges.read_text())

## The panji

Each session and each commit that changed one file. Newest first.

In [ ]:
for r in blend('charges.py', home=d/'.panjika', start=d):
    who = r.get('short') if r.kind == 'commit' else f"{r.session} {r.harness}"
    print(f"{r.kind:<8} {who:<24} {r.title[:58]}")

## Did it go into git

panjika compares the lines of each session with `git blame`.

In [ ]:
for sid in ('cc-4f21', 'rb-0c7e'):
    for v in landed(sid, home=d/'.panjika', start=d): print(v.line())

The lines from Monday are still there. The lines from Wednesday are not. The verdict gives the commit and the person that own those lines now. An agent that wants to add the rounding again reads this and stops.

## The log

In [ ]:
led = Ledger(d/'.panjika')
for r in led.sessions():
    row = led.session(r.session)
    print(f"{r.session}  {r.harness}/{r.model}  {row.n_steps} steps, {row.steps_fail} failed")
    print(f"    {r.prompt[:68]}")
    print(f"    {', '.join(t.path for t in row.files)}")

## On the disk

You commit `ledger/`, and it holds no source code. You do not commit `detail/`. It holds the full arguments, the full output, and the line hashes. `landed` needs the line hashes.

In [ ]:
print((d/'.panjika'/'.gitattributes').read_text())
print(Ledger(d/'.panjika').stats())

panjika never changes a record. A session is each record with its session id, in time order. Thus three processes can add the start, the tool calls and the end. They need no lock.